## Part 1 - Multiple Choice Question

In [4]:
# Import library
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [25]:
# Load dataset
orders_id = "1NXlSyuUYk6vPFZp3I_jD1PCaXAS7SVjI"
orders = pd.read_csv(f"https://drive.google.com/uc?export=download&id={orders_id}")

products_id = "1Uqvm7rLNAiSZ17cKWCmaUVEnuMvb49mK"
products = pd.read_csv(f"https://drive.google.com/uc?export=download&id={products_id}")

returns_id = "1vvvFFbTtsecRQ4HEGfnwor82n8mr1m_Y"
returns = pd.read_csv(f"https://drive.google.com/uc?export=download&id={returns_id}")

web_traffic_id = "1kbRN-wqNHxu5PhwtKAK2T7thiA3fMSK0"
web_traffic = pd.read_csv(f"https://drive.google.com/uc?export=download&id={web_traffic_id}")

order_items_id = "1liwGes0w4dL0Ml2-FJRAli5g-tQrrrDZ"
order_items = pd.read_csv(f"https://drive.google.com/uc?export=download&id={order_items_id}")

customers_id = "1jydC5-X5e-zMptdiSY8Bt7EmFdUxHHdj"
customers = pd.read_csv(f"https://drive.google.com/uc?export=download&id={customers_id}")

geography_id = "1WITg4fuiMrFMAjAwY5ISEBVL6HF6hoNz"
geography = pd.read_csv(f"https://drive.google.com/uc?export=download&id={geography_id}")

payments_id = "1_5jyV_qmqQsuqRTnx9szJ50k-RFy3avt"
payments = pd.read_csv(f"https://drive.google.com/uc?export=download&id={payments_id}")

sales_id = '1IQwZN6qJ6inKuYCnydSm5KyQ4oM7mDS6'
sales = pd.read_csv(f"https://drive.google.com/uc?export=download&id={sales_id}")

## **`Question 1`**
Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

In [7]:
orders["order_date"] = pd.to_datetime(orders["order_date"])
# Sắp xếp theo từng khách hàng và thời gian mua
orders_sorted = orders.sort_values(["customer_id", "order_date"])

# Tính số ngày giữa 2 lần mua liên tiếp của cùng 1 khách hàng
orders_sorted["inter_order_gap"] = (
    orders_sorted
    .groupby("customer_id")["order_date"]
    .diff()
    .dt.days
)

# Chỉ lấy các khoảng cách hợp lệ, tức là khách hàng có ít nhất 2 đơn
gaps = orders_sorted["inter_order_gap"].dropna()

median_gap = gaps.median()

print("Median inter-order gap:", median_gap)

Median inter-order gap: 144.0


> ***Đáp án cho câu 1: `C. 144.0`***

## **`Question 2`**

Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
trung bình cao nhất, với công thức (price − cogs)/price?


In [9]:
products["gross_margin_rate"] = (products["price"] - products["cogs"])/ products["price"]
segment_margin = products.groupby("segment", as_index= False)["gross_margin_rate"].mean().sort_values("gross_margin_rate", ascending = False)
print(segment_margin)
# Segment có tỷ suất lợi nhuận gộp trung bình cao nhất
best_segment = segment_margin.iloc[0]

print("Segment cao nhất:", best_segment["segment"])

       segment  gross_margin_rate
6     Standard           0.313442
5      Premium           0.285377
1  All-weather           0.284176
0   Activewear           0.265600
4  Performance           0.263650
2     Balanced           0.258038
7       Trendy           0.240758
3     Everyday           0.236343
Segment cao nhất: Standard


> ***Đáp án câu 2: `D. Standard`***

## **`Question 3`**
Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?







In [10]:
returns_products = pd.merge(returns, products, on = "product_id", how = "inner")
returns_products[returns_products["category"] == "Streetwear"].groupby("return_reason").size().idxmax()

'wrong_size'

> ***Đáp án câu 3: `B. wrong_size`***

## **`Question 4`**
Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [11]:
web_traffic.groupby("traffic_source")["bounce_rate"].mean().idxmin()

'email_campaign'

> ***Đáp án câu 4: `C. email_campaign`***

## **`Question 5`**
Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id
không null) xấp xỉ là bao nhiêu?

In [12]:
1 - order_items["promo_id"].isna().sum() / len(order_items)

np.float64(0.3866349316956521)

> ***Đáp án câu 5: `C. 0.39`***

## **`Question 6`**
Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số
đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong
nhóm)

In [20]:
customers_valid = customers.dropna(subset = ["age_group"])

orders_customers = pd.merge(customers_valid, orders, on = "customer_id", how = "left")

result = orders_customers.groupby("age_group").agg(
    total_orders=('order_id', 'count'),
    total_customers=('customer_id', 'nunique')
)

(result["total_orders"] / result["total_customers"]).idxmax()



'55+'

> ***Đáp án câu 6: `A. 55+`***

## **`Question 7`**
Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong
sales_train.csv?

In [30]:
geography_sales = sales.rename(columns = {"Date": "order_date"}).merge(orders[["order_date", "zip"]], on = "order_date").merge(geography[["zip", "region"]])
geography_sales.groupby("region")["Revenue"].sum().idxmax()

'East'

> ***Đáp án câu 7: `C. East`***

## **`Question 8`**
Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức
thanh toán nào được sử dụng nhiều nhất?

In [31]:
orders[orders["order_status"] == "cancelled"].groupby("payment_method")["payment_method"].count().idxmax()

'credit_card'

> ***Đáp án câu 8: `A. credit_card`***

## **`Question 9`**
Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao
nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join
với products theo product_id)?

In [32]:
returns_products = pd.merge(returns, products, on = "product_id", how = "inner")
returns_each_size = returns_products.groupby("size")["size"].count()

order_items_products = pd.merge(order_items, products, on = "product_id", how = "left")
orders_each_size = order_items_products.groupby("size")["size"].count()
(returns_each_size / orders_each_size).idxmax()


'S'

> ***Đáp án câu 9: `A. S`***

## **`Question 10`**
Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
mỗi đơn hàng cao nhất?

In [33]:
payments.groupby("installments")["payment_value"].mean().idxmax().item()

6


> ***Đáp án câu 10: `C. 6`***
